In [1]:
import json
import pandas as pd
import re
import os
import sklearn_crfsuite
from sklearn_crfsuite import metrics
from sklearn.model_selection import train_test_split
import random


In [2]:
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

DATA_PATH = os.path.join(parent_dir, 'data', 'synthetic_dataset.json')
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)
df = pd.DataFrame(data)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 529 entries, 0 to 528
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   text           529 non-null    str  
 1   event_type     529 non-null    str  
 2   address        437 non-null    str  
 3   is_actionable  529 non-null    bool 
 4   is_planned     529 non-null    bool 
dtypes: bool(2), str(3)
memory usage: 13.6 KB


In [3]:
data_with_addr = df[df["address"].notna()]
data_no_addr = df[df["address"].isna()]

print(f"Examples with address: {len(data_with_addr)}")
print(f"Examples without address: {len(data_no_addr)}")
print(f"Total for training: {len(data)}")

Examples with address: 437
Examples without address: 92
Total for training: 529


# bio tags

In [4]:
def normalize(s):
    s = s.lower()
    s = s.replace("\u2019", "'").replace("\u2018", "'").replace("`", "'")
    return s

def tokenize(text):
    tokens = []
    for raw in text.split():
        if re.match(r'^[а-яіїєґА-ЯІЇЄҐ]+\.$', raw):
            tokens.append(raw)
            continue
        if re.match(r'^[а-яіїєґА-ЯІЇЄҐ]+-[а-яіїєґА-ЯІЇЄҐ]+$', raw):  # р-н, б-р
            tokens.append(raw)
            continue

        m = re.match(
            r'^([^а-яіїєґА-ЯІЇЄҐa-zA-Z0-9\-]*)'
            r'([а-яіїєґА-ЯІЇЄҐa-zA-Z0-9][а-яіїєґА-ЯІЇЄҐa-zA-Z0-9\-\'\.]*)?'
            r'([^а-яіїєґА-ЯІЇЄҐa-zA-Z0-9\-]*)$',
            raw
        )
        if m:
            lead, word, trail = m.group(1) or '', m.group(2) or '', m.group(3) or ''
            # Separate leading chars individually (emoji, punct)
            for ch in lead:
                if ch.strip():
                    tokens.append(ch)
            if word:
                if word.endswith('.'):
                    if re.match(r'^\d[\d\-–]*$', word[:-1]):
                        tokens.append(word[:-1])
                        tokens.append('.')
                        continue
                    tokens.append(word)
                else:
                    tokens.append(word)
            for ch in trail:
                if ch.strip():
                    tokens.append(ch)
        else:
            tokens.append(raw)

    return [t for t in tokens if t and t.strip()]


def stem_uk(word):
    """Return a set of stems for Ukrainian word."""
    w = normalize(word)
    stems = {w}

    endings = [
        # Adjective endings
        'ського', 'ській', 'ського', 'ській', 'ськими', 'ській', 'ському',
        'зького', 'зькій', 'зького',
        'цького', 'цькій',
        'нього', 'ньому', 'нього',
        'овий', 'ового', 'овому', 'овим', 'ових', 'ової', 'овою',
        'евий', 'евого',
        'євий', 'євого',
        'ний', 'ного', 'ному', 'ним', 'них', 'ній', 'ною',
        'нього', 'ньої',
        'яний', 'яного',
        # Noun endings
        'ської', 'ськими',
        'ська', 'ське', 'ський', 'ські',
        'івський', 'івська', 'івське', 'івські',
        'івського', 'івській',
        'андр', 'анів',
        'ського',
        'івці', 'івку', 'івка', 'івки',
        'щині', 'щина', 'щини',
        'івській', 'євській',
        'ові', 'ому', 'ого',
        'ець', 'ці',
        'ина', 'ині', 'ину', 'иною',
        'ені', 'ена',
        'ана', 'ані',
        'ою', 'ою',
        'ної', 'ньої',
        'ій', 'ій',
        'ій', 'ій',
        # Short endings last
        'і', 'а', 'я', 'у', 'ю', 'е', 'є', 'ї',
    ]

    for e in sorted(endings, key=len, reverse=True):
        if w.endswith(e) and len(w) - len(e) >= 3:
            base = w[:-len(e)]
            stems.add(base)
            
            stems.add(base + 'ськ')
            stems.add(base + 'зьк')
            break

    return stems

TOPONYM_VARIANTS = {
    'поділ': {'поділ', 'подолі', 'подолу', 'подолом', 'поділу'},
    'оболонь': {'оболонь', 'оболоні', 'оболоню'},
    'троєщина': {'троєщина', 'троєщині', 'троєщину', 'троєщиною'},
    'борщагівка': {'борщагівка', 'борщагівці', 'борщагівку', 'борщагівкою'},
    'деміївка': {'деміївка', 'деміївці', 'деміївку'},
    'голосіїв': {'голосіїв', 'голосієві', 'голосієво', 'голосіїві'},
    'солом\'янка': {'солом\'янка', 'солом\'янці', 'солом\'янку'},
    'лук\'янівка': {'лук\'янівка', 'лук\'янівці', 'лук\'янівку'},
    'святошин': {'святошин', 'святошині', 'святошину'},
    'дарниця': {'дарниця', 'дарниці', 'дарницю'},
    'русанівка': {'русанівка', 'русанівці', 'русанівку'},
    'виноградар': {'виноградар', 'виноградарі', 'виноградарю'},
    'позняки': {'позняки', 'позняках', 'позняки'},
    'лівобережна': {'лівобережна', 'лівобережній', 'лівобережну'},
    'печерськ': {'печерськ', 'печерську', 'печерська'},
    'відрадний': {'відрадний', 'відрадному', 'відрадного', 'відрадна'},
}

# Build reverse: any variant -> canonical
VARIANT_TO_CANONICAL = {}
for canon, variants in TOPONYM_VARIANTS.items():
    for v in variants:
        VARIANT_TO_CANONICAL[v] = canon

def tokens_match(tok_norm, ref_norm):
    if tok_norm == ref_norm:
        return True
    # Prefix match (at least 5 chars common stem)
    tok_stripped = tok_norm.rstrip('.')
    ref_stripped = ref_norm.rstrip('.')
    if tok_stripped == ref_stripped:
        return True
    
    # Also match if one is a prefix of a range: "32" matches "32-50"
    if re.match(r'^\d+$', tok_stripped) and re.match(r'^\d+[\-–]', ref_stripped):
        if ref_stripped.startswith(tok_stripped):
            return True

    min_len = min(len(tok_norm), len(ref_norm))
    if min_len >= 5 and tok_norm[:min_len] == ref_norm[:min_len]:
        return True
    if min_len >= 4 and tok_norm[:4] == ref_norm[:4]:
        # Only if both are long enough to avoid false matches
        if len(tok_norm) >= 5 and len(ref_norm) >= 5:
            return True
    
    # Toponym variant dict
    tok_canon = VARIANT_TO_CANONICAL.get(tok_norm)
    ref_canon = VARIANT_TO_CANONICAL.get(ref_norm, ref_norm)
    if tok_canon and (tok_canon == ref_canon or tok_canon == ref_norm):
        return True
    
    # Stem match
    tok_stems = stem_uk(tok_norm)
    ref_stems = stem_uk(ref_norm)
    if tok_stems & ref_stems - {'', 'ськ', 'зьк', 'цьк'}:
        return True
    return False

def is_word(tok):
    return bool(re.search(r'[а-яіїєґА-ЯІЇЄҐa-zA-Z0-9]', tok))

def is_addr_prefix(tok_norm):
    prefixes = {'вул.', 'просп.', 'бульв.', 'пров.', 'пл.', 'р-н', 'ст.', 'м.', 'б-р', 'наб.', 
                'пр-т', 'пр.', 'вулиця', 'проспект', 'площа', 'провулок', 'бульвар', 'узвіз', 
                'острів', 'шосе', 'траса', 'метро', 'мікрорайон', 'зупинка', 'театр', 'комплекс', 
                'квартал', 'район', 'вокзал', 'парк', 'міст'}
    return tok_norm in prefixes


def find_sequence_fuzzy(token_norms, seq_norms, exclude_indices=None):
    """
    Find seq_norms as a window of consecutive WORD tokens in token_norms,
    skipping punctuation.
    """
    if exclude_indices is None:
        exclude_indices = set()

    word_indices = [i for i, t in enumerate(token_norms)
                    if (is_word(t) or re.match(r'^\d[\d\-–/]+$', t)) and i not in exclude_indices]
    m = len(seq_norms)
    if m == 0:
        return None

    for wi in range(len(word_indices) - m + 1):
        chunk = word_indices[wi:wi + m]
        if all(tokens_match(token_norms[chunk[j]], seq_norms[j]) for j in range(m)):
            # Return span covering chunk[0]..chunk[-1] including any punct in between
            return (chunk[0], chunk[-1])

    return None


def find_address_spans(tokens, address_field):
    token_norms = [normalize(t) for t in tokens]

    components = re.split(r'[,/]', address_field)
    spans = []
    used_indices = set()

    for comp in components:
        comp = comp.strip()
        if not comp:
            continue

        comp_tokens = tokenize(comp)
        comp_norms = [normalize(t) for t in comp_tokens if is_word(t)]

        #comp_norms_clean = [t.rstrip('.') if re.match(r'^\d[\d\-–]+\.?$', t) else t
        #            for t in comp_norms]
        comp_norms_clean = comp_norms
        if not comp_norms_clean:
            continue

        span = find_sequence_fuzzy(token_norms, comp_norms_clean, used_indices)
        if span:
            spans.append(span)
            used_indices.update(range(span[0], span[1]+1))
            continue


        stripped = comp_norms_clean[:]
        while stripped and is_addr_prefix(stripped[0]):
            stripped = stripped[1:]

        if len(stripped) >= 2 and stripped[0] in {'ст.', 'станція'} and stripped[1] == 'м.':
            stripped = stripped[2:]

        if stripped and stripped != comp_norms_clean:
            span = find_sequence_fuzzy(token_norms, stripped, used_indices)
            if span:
                spans.append(span)
                used_indices.update(range(span[0], span[1]+1))
                continue

        # Try individual long words
        for tok_norm in comp_norms_clean:
            if len(tok_norm) >= 4 and not tok_norm.endswith('.'):
                for i, tnorm in enumerate(token_norms):
                    if i not in used_indices and is_word(tnorm) and tokens_match(tnorm, tok_norm):
                        spans.append((i, i))
                        used_indices.add(i)
                        break

    return spans

def spans_to_bio(tokens, spans):
    addr_set = set()
    for start, end in spans:
        for i in range(start, end+1):
            addr_set.add(i)

    result = []
    prev_addr = False
    for i, tok in enumerate(tokens):
        if i in addr_set:
            tag = 'I-ADDR' if prev_addr else 'B-ADDR'
            prev_addr = True
        else:
            tag = 'O'
            prev_addr = False
        result.append([tok, tag])
    return result


def process_item(item):
    tokens = tokenize(item['text'])
    spans = find_address_spans(tokens, item['address'])
    bio = spans_to_bio(tokens, spans)
    return bio


data_with_addr2 = [item for item in data if item.get("address")]
all_sequences = [process_item(item) for item in data_with_addr2]


def fix_bio_sequence(seq: list[list[str, str]]) -> list[tuple[str, str]]:
    tokens = [t for t, _ in seq]
    tags   = [tag for _, tag in seq]

    MERGE_TOKENS = {",", ".", "–", "-", "—", "/", ";"}

    i = 0
    while i < len(tags):
        if tags[i] in ("B-ADDR", "I-ADDR"):
            j = i + 1
            while j < len(tags) and tags[j] == "O":
                if tokens[j] in MERGE_TOKENS:
                    j += 1  
                else:
                    break
            if j < len(tags) and tags[j] == "B-ADDR":
                for k in range(i + 1, j + 1):
                    tags[k] = "I-ADDR"
            i = j
        else:
            i += 1

    return list(zip(tokens, tags))

all_sequences = [fix_bio_sequence(seq) for seq in all_sequences]

In [27]:
DATA_BIO_PATH = os.path.join(parent_dir, 'data', 'data_with_tags.json')
seq_index = 0

for record in data:
    if record.get('address'):
        if seq_index < len(all_sequences):
            record['bio_tags'] = all_sequences[seq_index]
            seq_index += 1
        else:
            record['bio_tags'] = []
    else:
        record['bio_tags'] = []

print(f"{seq_index}/{len(all_sequences)}")

with open(DATA_BIO_PATH, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

437/437


# crf

In [6]:
DATA_BIO_PATH = os.path.join(parent_dir, 'data', 'data_with_tags.json')
with open(DATA_BIO_PATH, 'r', encoding='utf-8') as f:
    data_bio = json.load(f)
df_bio = pd.DataFrame(data_bio)
df_bio.info()
all_sequences = df_bio[df_bio["address"].notnull()]["bio_tags"].tolist()

<class 'pandas.DataFrame'>
RangeIndex: 529 entries, 0 to 528
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   text           529 non-null    str   
 1   event_type     529 non-null    str   
 2   address        437 non-null    str   
 3   is_actionable  529 non-null    bool  
 4   is_planned     529 non-null    bool  
 5   bio_tags       529 non-null    object
dtypes: bool(2), object(1), str(3)
memory usage: 17.7+ KB


In [7]:
len(all_sequences)

437

In [8]:
STREET_ABBRS = {
    "вул", "вул.", "вулиця", "просп", "просп.", "проспект",
    "пров", "пров.", "провулок", "бул", "бул.", "бульвар",
    "пл", "пл.", "площа", "пр-т", "шосе", "траса"
}
ADDR_TRIGGERS = {
    "парк", "парку", "міст", "мосту", "метро", "район",
    "районі", "зупинка", "зупинці", "вокзал", "вокзалу"
}
ADDR_CONTEXT = STREET_ABBRS | ADDR_TRIGGERS

def word2features(seq: list[tuple[str, str]], i: int) -> dict:
    token = seq[i][0]
    w = token.lower().rstrip(".,")

    features = {
        "bias": 1.0,

        "w.lower":    w,
        "w.suffix2":  w[-2:] if len(w) >= 2 else w,
        "w.suffix3":  w[-3:] if len(w) >= 3 else w,
        "w.prefix3":  w[:3]  if len(w) >= 3 else w,

        "w.isupper":  token.isupper(),
        "w.istitle":  token.istitle(),   
        "w.isdigit":  token.isdigit(),
        "w.isalpha":  token.isalpha(),
        "w.has_dash": "–" in token or "-" in token,  

        "w.is_street_abbr":   w in STREET_ABBRS,
        "w.is_addr_trigger":  w in ADDR_TRIGGERS,
        "w.is_house_num":     bool(re.match(r"^\d+[а-яa-z]?(/\d+)?$", token)),
        "w.is_house_range":   bool(re.match(r"^\d+[–\-]\d+", token)),  # 33–49
        "w.is_punct":         not token[0].isalnum() if token else False,
    }

    # token (i-1)
    if i > 0:
        pt, _ = seq[i - 1]
        pw = pt.lower().rstrip(".,")
        features.update({
            "-1:w.lower":         pw,
            "-1:w.istitle":       pt.istitle(),
            "-1:w.is_abbr":       pw in STREET_ABBRS,
            "-1:w.is_trigger":    pw in ADDR_TRIGGERS,
            "-1:w.is_addr_ctx":   pw in ADDR_CONTEXT,
        })
    else:
        features["BOS"] = True

    # (i-2)
    if i > 1:
        ppt, _ = seq[i - 2]
        ppw = ppt.lower().rstrip(".,")
        features.update({
            "-2:w.lower":       ppw,
            "-2:w.is_addr_ctx": ppw in ADDR_CONTEXT,
        })

    # (i+1)
    if i < len(seq) - 1:
        nt, _ = seq[i + 1]
        nw = nt.lower().rstrip(".,")
        features.update({
            "+1:w.lower":        nw,
            "+1:w.istitle":      nt.istitle(),
            "+1:w.is_house_num": bool(re.match(r"^\d+[а-яa-z]?(/\d+)?$", nt)),
            "+1:w.is_addr_ctx":  nw in ADDR_CONTEXT,
        })
    else:
        features["EOS"] = True

    # (i+2)
    if i < len(seq) - 2:
        nnt, _ = seq[i + 2]
        nnw = nnt.lower().rstrip(".,")
        features.update({
            "+2:w.lower":    nnw,
            "+2:w.istitle":  nnt.istitle(),
        })

    if i > 0:
        features["-1:w+ w"] = pw + "+" + w          # "вул.+антонова"
    if i < len(seq) - 1:
        features["+1:w+ w"] = w + "+" + nw


    if i < len(seq) - 1 and bool(re.match(r"^\d", nt)):
        features["+1:house_after"] = True


    features["w.ukr_street_end"] = any(w.endswith(suf) for suf in
        ["ська", "ький", "ька", "ьке", "ього", "ого", "а", "я"])

    return features

def seq2features(seq):
    return [word2features(seq, i) for i in range(len(seq))]

def seq2labels(seq):
    return [tag for _, tag in seq]


In [9]:
X = [seq2features(seq) for seq in all_sequences]
y = [seq2labels(seq)   for seq in all_sequences]
indices = list(range(len(X)))

X_train, X_test, y_train, y_test, train_indices, test_indices = train_test_split(
    X, y, indices, test_size=0.2, random_state=42)

In [10]:
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.1, c2=0.05,
    max_iterations=200,
    all_possible_transitions=True
)
crf.fit(X_train, y_train)
y_pred = crf.predict(X_test)

#token-level scores
print(metrics.flat_classification_report(y_test, y_pred, labels=["B-ADDR","I-ADDR"], digits=3))

              precision    recall  f1-score   support

      B-ADDR      0.893     0.793     0.840       116
      I-ADDR      0.889     0.912     0.900       193

   micro avg      0.890     0.867     0.879       309
   macro avg      0.891     0.853     0.870       309
weighted avg      0.891     0.867     0.878       309



In [11]:
def extract_entities(tokens, tags):
    entities = []
    start = None
    for i, (tok, tag) in enumerate(zip(tokens, tags)):
        if tag == "B-ADDR":
            if start is not None:
                entities.append((start, i))
            start = i
        elif tag == "O" and start is not None:
            entities.append((start, i))
            start = None
    if start is not None:
        entities.append((start, len(tokens)))
    return set(entities) 

def entity_level_scores(X_test, y_test, y_pred, test_indices_seqs):
    tp = fp = fn = 0

    for seq, true_tags, pred_tags in zip(test_indices_seqs, y_test, y_pred):
        tokens = [t for t, _ in seq]
        true_ents = extract_entities(tokens, true_tags)
        pred_ents = extract_entities(tokens, pred_tags)

        tp += len(true_ents & pred_ents)   
        fp += len(pred_ents - true_ents)   
        fn += len(true_ents - pred_ents)  

    precision = tp / (tp + fp) if tp + fp > 0 else 0
    recall = tp / (tp + fn) if tp + fn > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    
    print(f"Precision: {precision:.3f}")
    print(f"Recall: {recall:.3f}")
    print(f"F1: {f1:.3f}")
    print(f"TP={tp} FP={fp} FN={fn}")

entity_level_scores(X_test, y_test, y_pred, [all_sequences[i] for i in test_indices])

Precision: 0.777
Recall: 0.690
F1: 0.731
TP=80 FP=23 FN=36


In [12]:
def error_analysis(y_test, y_pred, test_indices_seqs, n=20):
    errors = []

    for seq, true_tags, pred_tags in zip(test_indices_seqs, y_test, y_pred):
        if true_tags == pred_tags:
            continue 

        tokens = [t for t, _ in seq]
        true_ents = extract_entities(tokens, true_tags)
        pred_ents = extract_entities(tokens, pred_tags)

        if not pred_ents and true_ents:
            error_type = "MISSED"   
        elif pred_ents and not true_ents:
            error_type = "FALSE_POS"   
        elif true_ents != pred_ents:
            error_type = "WRONG_SPAN" 
        else:
            continue

        errors.append({
            "type": error_type,
            "text": " ".join(tokens),
            "true": [" ".join(tokens[s:e]) for s, e in true_ents],
            "pred": [" ".join(tokens[s:e]) for s, e in pred_ents],
        })

    print(f"Count of errors: {len(errors)}")
    counts = {}
    for e in errors:
        counts[e["type"]] = counts.get(e["type"], 0) + 1
    print(f"Error distribution: {counts}\n")

    for e in random.sample(errors, min(n, len(errors))):
        print(f"[{e['type']}]")
        print(f"Text: {e['text'][:120]}")
        print(f"Real: {e['true']}")
        print(f"Model: {e['pred']}")

error_analysis(y_test, y_pred, [all_sequences[i] for i in test_indices])

Count of errors: 28
Error distribution: {'WRONG_SPAN': 27, 'MISSED': 1}

[WRONG_SPAN]
Text: КП «Київенерго» повідомляє про планову заміну силового кабелю та відключення електропостачання для мешканців вул. Братис
Real: ['вул. Братиславської, 66–82,', 'Деснянський район,']
Model: ['вул. Братиславської, 66–82,']
[WRONG_SPAN]
Text: КМДА інформує про проведення Дня Києва 25–26 травня 2024 року. Святкові заходи відбуватимуться на вул. Хрещатик, Хрещато
Real: ['набережній Дніпра', 'вул. Хрещатик,', 'Хрещатому парку,']
Model: ['вул. Хрещатик, Хрещатому парку, набережній Дніпра']
[WRONG_SPAN]
Text: Буря поламала кілька дерев на Русанівці — вул. Ентузіастів. Один автомобіль пошкоджено.
Real: ['Русанівці — вул. Ентузіастів.']
Model: ['вул. Ентузіастів.', 'Русанівці']
[WRONG_SPAN]
Text: ⚠️ Зараз стоїть все Бориспільське шосе від Харківської до Бортничів — аварія.
Real: ['Бортничів', 'Бориспільське шосе', 'Харківської']
Model: ['Бориспільське шосе від Харківської до Бортничів']
[WRONG_SPAN]
Text: 

In [13]:
import re
import random
from collections import defaultdict

def normalize_address(text: str) -> str:
    text = text.lower()
    text = re.sub(r'[.,№"\'`–-——]', '', text)       
    text = re.sub(r'\s+', ' ', text).strip()
    text = text.replace(' та ', ' ').replace(' і ', ' ').replace(' та ', ' ')
    return text

def jaccard_similarity(a: str, b: str) -> float:
    if not a or not b:
        return 0.0
    set_a = set(a.split())
    set_b = set(b.split())
    intersection = len(set_a & set_b)
    union = len(set_a | set_b)
    return intersection / union if union > 0 else 0.0


def relaxed_error_analysis(y_test, y_pred, test_indices_seqs, threshold: float = 0.85, n: int = 20):
    errors = []
    relaxed_matches = 0
    total_compared = 0

    for seq_idx, (seq, true_tags, pred_tags) in enumerate(zip(test_indices_seqs, y_test, y_pred)):
        tokens = [t for t, _ in seq]
        text = " ".join(tokens)

        true_ents = extract_entities(tokens, true_tags)
        pred_ents = extract_entities(tokens, pred_tags)

        true_texts = [normalize_address(" ".join(tokens[s:e])) for s, e in true_ents]
        pred_texts = [normalize_address(" ".join(tokens[s:e])) for s, e in pred_ents]
        exact_match = true_ents == pred_ents

        matched = set()
        relaxed_correct = []
        for p_text in pred_texts:
            for j, t_text in enumerate(true_texts):
                if j in matched:
                    continue
                if jaccard_similarity(p_text, t_text) >= threshold:
                    matched.add(j)
                    relaxed_correct.append((true_texts[j], p_text))
                    break

        total_compared += len(true_ents) + len(pred_ents)

        if exact_match and len(true_ents) == len(pred_ents):
            continue  

        if not pred_ents and true_ents:
            error_type = "MISSED"
        elif pred_ents and not true_ents:
            error_type = "FALSE_POS"
        elif len(matched) == len(true_ents) and len(matched) == len(pred_ents):
            error_type = "RELAXED_CORRECT"   
        else:
            error_type = "WRONG_SPAN"      

        errors.append({
            "type": error_type,
            "seq_idx": seq_idx,
            "text": text[:200] + ("..." if len(text) > 200 else ""),
            "true": [" ".join(tokens[s:e]) for s, e in true_ents],
            "pred": [" ".join(tokens[s:e]) for s, e in pred_ents],
            "true_norm": true_texts,
            "pred_norm": pred_texts,
            "relaxed_pairs": relaxed_correct,
            "jaccard_scores": [jaccard_similarity(p, t) for p in pred_texts for t in true_texts]
        })

        if error_type == "RELAXED_CORRECT":
            relaxed_matches += 1

    print(f"Count of examples with errors (exact): {len(errors)}")
    print(f"Of which 'essentially correct' (RELAXED_CORRECT): {relaxed_matches}")
    print(f"Real errors: {len(errors) - relaxed_matches}\n")

    counts = {}
    for e in errors:
        counts[e["type"]] = counts.get(e["type"], 0) + 1
    print(f"Distribution of error types:")
    for etype, cnt in sorted(counts.items()):
        print(f"  {etype:15} : {cnt}")

    print(f"\nExamples of errors (shown {min(n, len(errors))} out of {len(errors)}):")
    sample_errors = random.sample(errors, min(n, len(errors)))

    for e in sample_errors:
        print(f"[{e['type']}]")
        print(f"Text: {e['text']}")
        
        if e['true']:
            print(f"Correct address: {e['true']}")
        if e['pred']:
            print(f"Model extracted: {e['pred']}")
        
        if e['type'] == "RELAXED_CORRECT" and e['relaxed_pairs']:
            print(f"(Jaccard ≥ {threshold}):")
            for true_norm, pred_norm in e['relaxed_pairs']:
                print(f"   '{true_norm}'  ≈  '{pred_norm}'")
        
        elif e['type'] in ["WRONG_SPAN", "MISSED", "FALSE_POS"]:
            if e.get('true_norm') and e.get('pred_norm'):
                print(f"Normalized:")
                print(f"   True: {e['true_norm']}")
                print(f"   Pred: {e['pred_norm']}")
        
        print("-" * 80)
relaxed_error_analysis(y_test, y_pred, [all_sequences[i] for i in test_indices], threshold=0.6)

Count of examples with errors (exact): 28
Of which 'essentially correct' (RELAXED_CORRECT): 5
Real errors: 23

Distribution of error types:
  MISSED          : 1
  RELAXED_CORRECT : 5
  WRONG_SPAN      : 22

Examples of errors (shown 20 out of 28):
[RELAXED_CORRECT]
Text: Теремки, вул. Теремківська — нічний клуб у підвалі гатить до 6 ранку щоп'ятниці. Вже місяць.
Correct address: ['Теремки, вул. Теремківська']
Model extracted: ['вул. Теремківська']
(Jaccard ≥ 0.6):
   'теремки вул теремківська'  ≈  'вул теремківська'
--------------------------------------------------------------------------------
[RELAXED_CORRECT]
Text: ❄️ На мосту Метро ожеледь — кілька авто вже зіткнулись. Уникайте або їдьте дуже повільно.
Correct address: ['мосту Метро']
Model extracted: ['мосту Метро ожеледь']
(Jaccard ≥ 0.6):
   'мосту метро'  ≈  'мосту метро ожеледь'
--------------------------------------------------------------------------------
[RELAXED_CORRECT]
Text: КМДА повідомляє про відкриття нового культу

In [14]:
def extract_spans(tags: list[str], tokens: list[str]) -> list[str]:
    spans, buf = [], []
    for token, tag in zip(tokens, tags):
        if tag == "B-ADDR":
            if buf:
                spans.append(" ".join(buf))
            buf = [token]
        elif tag == "I-ADDR" and buf:
            buf.append(token)
        else:
            if buf:
                spans.append(" ".join(buf))
                buf = []
    if buf:
        spans.append(" ".join(buf))
    return spans

demo_texts = [
    "Прорив труби на вул. Оболонська 15, без води половина будинку",
    "Ожеледиця на дорогах Троєщини, комунальники працюють",
    "Концерт у парку Шевченка сьогодні ввечері",
    "ДТП на перехресті просп. Перемоги і вул. Дорогожицька",
    "Повідомляємо, що КК «Київавтодор» 3 червня 2024 року з 07:00 до 20:00 здійснюватиме планові роботи з реконструкції дорожньої розв'язки на перетині просп. Бандери та вул. Дорогожицької, Шевченківський район. Частковий об'їзд організовано через вул. Олени Теліги.",
    "Ожеледиця на мосту Патона зранку — кілька ДТП вже сталось. Їдьте обережно!",
    "Завтра на вул. Кирилівській у Києві частково обмежать рух.\n\nЗа даними «Київавтодору», обмеження діятимуть з 8:00 до 17:00 на ділянці від вул. Петропавлівської до вул. Білицької. Там проводитимуть ремонт дорожнього покриття, рух перекриватимуть поетапно.",
    "У квітні 2026 року в Києві презентують «Шолом пам’яті» Владислава Гераскевича.\n\nОригінальний шолом і новий художній варіант зображають 22 загиблих маріупольських спортсменів, серед них 8 дітей. Ідею втілила художниця Ірина Проць. Це пам’ять про жертв російської агресії.\n\nВиставка відбудеться в Музеї «Голоси Мирних» (Київ, вул. Богдана Хмельницького, 7, 1-й поверх Музею історії міста). Деталі — на сайті civilvoicesmuseum.org.",
]
def tokenize(text: str) -> list[str]:
    tokens = re.findall(r"[\w']+|[.,!?;:—\-]", text)
    return tokens

for text in demo_texts:
    tokens = tokenize(text)
    dummy_seq = [(tok, "O") for tok in tokens]
    features = seq2features(dummy_seq)
    pred = crf.predict([features])[0]
    addr = extract_spans(pred, tokens)
    print(f"{addr} {text[:40]}")

['вул . Оболонська 15'] Прорив труби на вул. Оболонська 15, без 
[] Ожеледиця на дорогах Троєщини, комунальн
[] Концерт у парку Шевченка сьогодні ввечер
['просп . Перемоги і вул . Дорогожицька'] ДТП на перехресті просп. Перемоги і вул.
['просп . Бандери та вул . Дорогожицької', 'вул . Олени Теліги'] Повідомляємо, що КК «Київавтодор» 3 черв
['мосту Патона зранку'] Ожеледиця на мосту Патона зранку — кільк
['вул . Кирилівській', 'вул . Петропавлівської до вул . Білицької'] Завтра на вул. Кирилівській у Києві част
['Києві', 'Музеї Голоси Мирних Київ , вул . Богдана Хмельницького , 7'] У квітні 2026 року в Києві презентують «
